# Things to analise
[x] Check if changes in code solved problems in v1.

In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve().parent

os.chdir(PROJECT_ROOT)

working_dir = os.getcwd()
print(working_dir)

C:\Users\Ocean\Desktop\Semestre 12 - TFG\ontometal


# Band data checks

In [5]:
bands = pd.read_csv("src/etl/data/normalized/bands_v2.csv", low_memory=False).copy()

In [6]:
# Bands with Na values
bands.isna().sum().sort_values(ascending=False)

producedBy         151166
releases             3791
bandId                  0
bandName                0
metalArchiveUrl         0
status                  0
hasGenre                0
hasCountry              0
dtype: int64

In [7]:
# Number of duplicated
bands.duplicated().sum()

np.int64(0)

In [8]:
# Number of bands with same bandId
bands.groupby("bandId").size().reset_index(name="count").query("count > 1").shape[0]

0

Theres no critical data or duplicated bands. Only have nulls on labels, which is expected.

# Bands join countries
Countries was already ok

In [9]:
bands = pd.read_csv("src/etl/data/normalized/bands_v2.csv", low_memory=False).copy()
countries = pd.read_csv("src/etl/data/normalized/countries_v2.csv").copy()
bands.merge(countries, left_on="hasCountry", right_on="id", how="left").drop(
    columns=["id"]
).rename(columns={"name": "countryName"})[["bandId", "countryName"]]

,bandId,countryName
0,3540442600,united states
1,3540525193,colombia
2,3540473101,united kingdom
3,3540535978,united states
4,3540352307,united states
...,...,...
182272,97695,"korea, south"
182273,3540435197,"korea, south"
182274,3540473622,japan
182275,3540493653,japan


In [10]:
# All bands has a country assigned
bands["hasCountry"].isna().sum()

np.int64(0)

# Bands join genres
We already know not all bands will have genres

In [13]:
# We merge with genre name because we ensure on etl to map genres
# Lets ensure genres are mapped correctly and if not, ensure there's no strange values in genres
bands = pd.read_csv("src/etl/data/normalized/bands_v2.csv", low_memory=False).copy()
genres = pd.read_csv("src/etl/data/normalized/genres_v2.csv").copy()
genres["name"] = genres["name"].astype(str).str.strip().str.lower()

bands["hasGenre"] = bands["hasGenre"].apply(
    lambda x: [g.strip() for g in str(x).split(",") if g.strip()] if pd.notna(x) else []
)

used = set(
    bands["hasGenre"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)


unused_genres = genres[~genres["name"].isin(used)]

print("Total genres:", len(genres))
print("Used genres:", len(used))
print("Unused genres:", len(unused_genres))

Total genres: 1375
Used genres: 1373
Unused genres: 2


In [14]:
unused_genres.head()

,id,name
458,459,experimental industrial doom
949,950,power metal with electronic elements


In [15]:
# Show the 2 unused genres and trace them back to raw data
print("Unused genres:")
print(unused_genres[["id", "name"]].to_string(index=False))
print()

raw_bands = pd.read_csv("src/etl/data/raw/metal_bands_roster.csv").copy()
for genre_name in unused_genres["name"].tolist():
    matches = raw_bands[raw_bands["Genre"].str.contains(genre_name, case=False, na=False)]
    print(f"\n--- Genre: '{genre_name}' ---")
    print(f"Found in {len(matches)} raw band(s)")
    if len(matches) > 0:
        print(matches[["Band ID", "Name", "Genre"]].head(5).to_string(index=False))

Unused genres:
 id                                 name
459         experimental industrial doom
950 power metal with electronic elements


--- Genre: 'experimental industrial doom' ---
Found in 1 raw band(s)
Band ID Name                                    Genre
      0  0N0 Experimental Industrial Doom/Death Metal

--- Genre: 'power metal with electronic elements' ---
Found in 1 raw band(s)
Band ID    Name                                Genre
      3 3lusion Power Metal with Electronic elements


C:\Users\Ocean\AppData\Local\Temp\ipykernel_12904\502871000.py:6: DtypeWarning: Columns (0: Band ID) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_bands = pd.read_csv("src/etl/data/raw/metal_bands_roster.csv").copy()


In [16]:
# Check if the bands that use these genres were dropped by normalization
raw_bands = pd.read_csv("src/etl/data/raw/metal_bands_roster.csv", low_memory=False).copy()
bands = pd.read_csv("src/etl/data/normalized/bands_v2.csv", low_memory=False).copy()

band_3lusion = raw_bands[raw_bands["Name"] == "3lusion"]
print("--- Band '3lusion' ---")
print(f"Raw Band ID: {band_3lusion['Band ID'].values}")
print(f"Exists in normalized? {band_3lusion['Band ID'].values[0] in bands['bandId'].values}")

band_0n0 = raw_bands[raw_bands["Name"] == "0N0"]
print(f"\n--- Band '0N0' ---")
print(f"Raw Band ID: {band_0n0['Band ID'].values}")
print(f"Exists in normalized? {band_0n0['Band ID'].values[0] in bands['bandId'].values}")


--- Band '3lusion' ---
Raw Band ID: [3]
Exists in normalized? False

--- Band '0N0' ---
Raw Band ID: [0]
Exists in normalized? False


2 genres unused due to belong to bands that normalize process deleted them removing duplicated ids, so we assume this as correct behavior.

# Bands join labels

There are duplicated values on labels raw dataset, field label ID, let's check that the resulting label corresponds to the value assinged on complete_roser dataset

In [17]:
# Check labels assigned to bands correspond to labels in raw labels dataset
complete_roster = pd.read_csv("src/etl/data/raw/complete_roster.csv").copy()
raw_labels = pd.read_csv("src/etl/data/raw/labels_roster.csv").copy()
labels = pd.read_csv("src/etl/data/normalized/labels_v2.csv").copy()

C:\Users\Ocean\AppData\Local\Temp\ipykernel_12904\1355232647.py:2: DtypeWarning: Columns (0: Band ID, 1: Name_y, 2: Specialization, 3: Status_y, 4: Country_y, 5: Website, 6: Online Shopping) have mixed types. Specify dtype option on import or set low_memory=False.
  complete_roster = pd.read_csv("src/etl/data/raw/complete_roster.csv").copy()


In [18]:
labels.isna().sum().sort_values(ascending=False)

hasSpecialization    7649
websiteUrl           7388
hasCountry           2277
status                  0
labelName               0
labelId                 0
producer                0
dtype: int64

In [20]:
# Check raw label from first label on normalized labels
label_name = labels["labelName"].iloc[0]
print("Label name:", label_name)

label_id = labels["labelId"].iloc[0]
print("Label ID:", label_id)

band_produced = [int(v) for v in str(labels["producer"].iloc[0]).split(",") if v.strip().isdigit()]
band_produced = band_produced[0] if band_produced else None

print("Band produced:", band_produced)

Label name: nuclear blast
Label ID: 2
Band produced: 213


The selected label still being nuclear blast which is the most complete record and in status active.

# Join with releases

In [22]:
releases = pd.read_csv("src/etl/data/normalized/releases_v2.csv").copy()
import ast

bands["releases"] = bands["releases"].apply(
    lambda x: [g.strip() for g in str(x).split(",") if g.strip()] if pd.notna(x) else []
)

In [23]:
releases.isna().sum().sort_values(ascending=False)

releaseId       0
releaseTitle    0
releaseYear     0
releaseType     0
releasedBy      0
dtype: int64

In [24]:
releases

,releaseId,releaseTitle,releaseYear,releaseType,releasedBy
0,1,butterfly,1989,ep,3540442600
1,2,things to come,1995,ep,3540442600
2,3,i can not believe,2022,single,3540525193
3,4,ritual magick,2023,ep,3540525193
4,5,purge of the weak and dying,2014,ep,3540473101
...,...,...,...,...,...
627113,627114,勿忘草,2004,single,3540493653
627114,627115,占有愛,2005,full-length,3540493653
627115,627116,first demo,1999,demo,37240
627116,627117,second demo,1999,demo,37240


In [27]:

bands = bands.explode("releases")
bands["releases"] = pd.to_numeric(bands["releases"], errors="coerce").astype("Int64")
releases["releaseId"] = pd.to_numeric(releases["releaseId"], errors="coerce").astype("Int64")
merged = bands.merge(
    releases,
    left_on="releases",
    right_on="releaseId",
    how="left"
)
merged["releasedBy"] = merged["releasedBy"].astype("Int64")
merged["releaseId"] = merged["releaseId"].astype("Int64")
merged.head(20)

,bandId,bandName,status,metalArchiveUrl,releases,producedBy,hasGenre,hasCountry,releaseId,releaseTitle,releaseYear,releaseType,releasedBy
0,3540442600,a // solution,split-up,https://www.metal-archives.com/bands/a_--_solu...,1,NaN,"crust punk with doom metal influences,thrash m...",1,1,butterfly,1989.0,ep,3540442600
1,3540442600,a // solution,split-up,https://www.metal-archives.com/bands/a_--_solu...,2,NaN,"crust punk with doom metal influences,thrash m...",1,2,things to come,1995.0,ep,3540442600
2,3540525193,a black cold diamond,active,https://www.metal-archives.com/bands/a_black_c...,3,54710.0,"heavy metal,doom metal",2,3,i can not believe,2022.0,single,3540525193
3,3540525193,a black cold diamond,active,https://www.metal-archives.com/bands/a_black_c...,4,54710.0,"heavy metal,doom metal",2,4,ritual magick,2023.0,ep,3540525193
4,3540473101,a billion limbs,active,https://www.metal-archives.com/bands/a_billion...,5,NaN,"groove metal,death metal with deathcore influe...",3,5,purge of the weak and dying,2014.0,ep,3540473101
5,3540473101,a billion limbs,active,https://www.metal-archives.com/bands/a_billion...,6,NaN,"groove metal,death metal with deathcore influe...",3,6,catalyst,2015.0,full-length,3540473101
6,3540473101,a billion limbs,active,https://www.metal-archives.com/bands/a_billion...,7,NaN,"groove metal,death metal with deathcore influe...",3,7,progression & constraint,2017.0,ep,3540473101
7,3540473101,a billion limbs,active,https://www.metal-archives.com/bands/a_billion...,8,NaN,"groove metal,death metal with deathcore influe...",3,8,adjusted trajectory,2018.0,full-length,3540473101
8,3540473101,a billion limbs,active,https://www.metal-archives.com/bands/a_billion...,9,NaN,"groove metal,death metal with deathcore influe...",3,9,absolute horror,2020.0,full-length,3540473101
9,3540535978,a band named jon,active,https://www.metal-archives.com/bands/a_band_na...,10,62768.0,"brutal death metal,death metal with grindcore",1,10,mind your fucking business,2022.0,demo,3540535978


In [28]:
# Check that bandId matches releasedBy on all merged rows (excluding NaN from left join)
matched = merged.dropna(subset=["releasedBy"])
mismatches = matched[matched["bandId"] != matched["releasedBy"]]
print(f"Total merged rows with release data: {len(matched)}")
print(f"Mismatches (bandId != releasedBy): {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches[["bandId", "releases", "releaseId", "releasedBy"]].head(10))

Total merged rows with release data: 627118
Mismatches (bandId != releasedBy): 0


# Summary of result in v2

1. No duplicates or critical null on bands data

3. Only 2 genres unused due to belong to duplicated ids bands

4.  Labels are being picked for completeness and status

5. Countries are ok

6. Releases without null titles

7. Releases joined by foreign key match on band releasedBy

In [29]:
# Data quality summary for RDF conversion viability
bands_check = pd.read_csv("src/etl/data/normalized/bands_v2.csv", low_memory=False)
countries_check = pd.read_csv("src/etl/data/normalized/countries_v2.csv")
genres_check = pd.read_csv("src/etl/data/normalized/genres_v2.csv")
labels_check = pd.read_csv("src/etl/data/normalized/labels_v2.csv")
releases_check = pd.read_csv("src/etl/data/normalized/releases_v2.csv")

print("=" * 60)
print("  DATA QUALITY SUMMARY FOR RDF CONVERSION")
print("=" * 60)

# --- Entity counts ---
print("\n--- Entity Counts ---")
print(f"  Bands:     {len(bands_check):,}")
print(f"  Countries: {len(countries_check):,}")
print(f"  Genres:    {len(genres_check):,}")
print(f"  Labels:    {len(labels_check):,}")
print(f"  Releases:  {len(releases_check):,}")

# --- Identifier integrity (critical for URI generation) ---
print("\n--- Identifier Integrity (URIs) ---")
print(f"  Bands    - NaN ID: {bands_check['bandId'].isna().sum()} | Duplicated ID: {bands_check['bandId'].duplicated().sum()}")
print(f"  Countries - NaN ID: {countries_check['id'].isna().sum()} | Duplicated ID: {countries_check['id'].duplicated().sum()}")
print(f"  Genres   - NaN ID: {genres_check['id'].isna().sum()} | Duplicated ID: {genres_check['id'].duplicated().sum()}")
print(f"  Labels   - NaN ID: {labels_check['labelId'].isna().sum()} | Duplicated ID: {labels_check['labelId'].duplicated().sum()}")
print(f"  Releases - NaN ID: {releases_check['releaseId'].isna().sum()} | Duplicated ID: {releases_check['releaseId'].duplicated().sum()}")

# --- Required literal properties (rdfs:label / hasName) ---
print("\n--- Required Literals (names/titles) ---")
print(f"  Bands without name:     {bands_check['bandName'].isna().sum()}")
print(f"  Countries without name: {countries_check['name'].isna().sum()}")
print(f"  Genres without name:    {genres_check['name'].isna().sum()}")
print(f"  Labels without name:    {labels_check['labelName'].isna().sum()}")
print(f"  Releases without title: {releases_check['releaseTitle'].isna().sum()}")

# --- Object property completeness (relationships) ---
print("\n--- Relationship Completeness ---")
print(f"  Bands with country (hasCountry):   {bands_check['hasCountry'].notna().sum():,} / {len(bands_check):,}")
print(f"  Bands with label (producedBy):     {bands_check['producedBy'].notna().sum():,} / {len(bands_check):,}")
bands_with_releases = (bands_check['releases'] != '[]').sum()
print(f"  Bands with releases:               {bands_with_releases:,} / {len(bands_check):,}")

bands_check['_genre_list'] = bands_check['hasGenre'].apply(
    lambda x: [g.strip() for g in str(x).split(",") if g.strip()] if pd.notna(x) else []
)
bands_with_genre = (bands_check['_genre_list'].apply(lambda x: len(x) > 0)).sum()
print(f"  Bands with genre (hasGenre):       {bands_with_genre:,} / {len(bands_check):,}")

print(f"  Releases with band (releasedBy):   {releases_check['releasedBy'].notna().sum():,} / {len(releases_check):,}")

# --- Foreign key validity (critical — do referenced IDs exist?) ---
print("\n--- Foreign Key Validity (Critical) ---")
valid_country_ids = set(countries_check['id'])
invalid_country_refs = bands_check['hasCountry'].dropna().apply(lambda x: int(x) not in valid_country_ids).sum()
print(f"  Bands -> Countries orphan refs:  {invalid_country_refs}")

valid_label_ids = set(labels_check['labelId'])
invalid_label_refs = bands_check['producedBy'].dropna().apply(lambda x: int(x) not in valid_label_ids).sum()
print(f"  Bands -> Labels orphan refs:     {invalid_label_refs}")

valid_band_ids = set(bands_check['bandId'])
invalid_release_refs = releases_check['releasedBy'].dropna().apply(lambda x: int(x) not in valid_band_ids).sum()
print(f"  Releases -> Bands orphan refs:   {invalid_release_refs}")

valid_genre_names = set(genres_check['name'].str.strip().str.lower())
all_band_genres = bands_check['_genre_list'].explode().dropna().astype(str).str.strip().str.lower()
invalid_genre_refs = (~all_band_genres.isin(valid_genre_names)).sum()
print(f"  Bands -> Genres orphan refs:     {invalid_genre_refs}")

bands_check.drop(columns=['_genre_list'], inplace=True)

# --- Foreign key validity (non-critical — Label relationships) ---
print("\n--- Foreign Key Validity (Non-Critical — Labels) ---")
labels_check['_producer_list'] = labels_check['producer'].apply(
    lambda x: [int(v) for v in str(x).split(",") if v.strip().isdigit()] if pd.notna(x) else []
)
all_label_bands = labels_check['_producer_list'].explode().dropna()
invalid_label_band_refs = (~all_label_bands.astype(int).isin(valid_band_ids)).sum()
print(f"  Labels -> Bands (producer) orphan refs:        {invalid_label_band_refs}")

invalid_label_country_refs = labels_check['hasCountry'].dropna().apply(lambda x: int(x) not in valid_country_ids).sum()
print(f"  Labels -> Countries (hasCountry) orphan refs:  {invalid_label_country_refs}")

labels_check['_spec_list'] = labels_check['hasSpecialization'].apply(
    lambda x: [g.strip() for g in str(x).split(",") if g.strip()] if pd.notna(x) else []
)
all_label_genres = labels_check['_spec_list'].explode().dropna().astype(str).str.strip().str.lower()
invalid_label_genre_refs = (~all_label_genres.isin(valid_genre_names)).sum()
print(f"  Labels -> Genres (hasSpecialization) orphan refs: {invalid_label_genre_refs}")

labels_check.drop(columns=['_producer_list', '_spec_list'], inplace=True)

# --- Status values (non-critical — will become individuals or literals) ---
print("\n--- Status Values (Non-Critical) ---")
band_status_nan = bands_check['status'].isna().sum()
band_status_vals = sorted(bands_check['status'].dropna().unique())
print(f"  bandStatus NaN: {band_status_nan}")
print(f"  bandStatus values ({len(band_status_vals)}): {band_status_vals}")

label_status_nan = labels_check['status'].isna().sum()
label_status_vals = sorted(labels_check['status'].dropna().unique())
print(f"  labelStatus NaN: {label_status_nan}")
print(f"  labelStatus values ({len(label_status_vals)}): {label_status_vals}")

# --- Release data properties (non-critical) ---
print("\n--- Release Data Properties (Non-Critical) ---")
release_year_nan = releases_check['releaseYear'].isna().sum()
print(f"  releaseYear NaN: {release_year_nan}")

# xsd:gYear validation: must be a 4-digit integer
release_years = releases_check['releaseYear'].dropna().astype(str)
valid_gyear = release_years.str.fullmatch(r'\d{4}')
invalid_gyear = (~valid_gyear).sum()
print(f"  releaseYear invalid xsd:gYear (not 4-digit): {invalid_gyear}")
if invalid_gyear > 0:
    print(f"    Sample invalid values: {release_years[~valid_gyear].unique()[:10].tolist()}")

release_type_nan = releases_check['releaseType'].isna().sum()
release_type_vals = sorted(releases_check['releaseType'].dropna().unique())
print(f"  releaseType NaN: {release_type_nan}")
print(f"  releaseType values ({len(release_type_vals)}): {release_type_vals}")

# --- Verdict ---
print("\n" + "=" * 60)
total_issues = (
    bands_check['bandId'].isna().sum() + bands_check['bandId'].duplicated().sum() +
    bands_check['bandName'].isna().sum() +
    invalid_country_refs + invalid_label_refs + invalid_release_refs + invalid_genre_refs
)
if total_issues == 0:
    print("  VERDICT: Data is READY for RDF conversion.")
    print("  All identifiers are unique and non-null.")
    print("  All critical foreign keys point to existing entities.")
    print("  All required literals are present.")
    non_critical = invalid_label_band_refs + invalid_label_country_refs + invalid_label_genre_refs + invalid_gyear + release_year_nan
    if non_critical > 0:
        print(f"\n  NOTE: {non_critical} non-critical issue(s) found.")
        print("  These can be handled during RDF conversion (skip NaN, create individuals).")
else:
    print(f"  VERDICT: {total_issues} critical issue(s) found. Review before RDF conversion.")

  DATA QUALITY SUMMARY FOR RDF CONVERSION

--- Entity Counts ---
  Bands:     182,277
  Countries: 158
  Genres:    1,375
  Labels:    13,538
  Releases:  627,118

--- Identifier Integrity (URIs) ---
  Bands    - NaN ID: 0 | Duplicated ID: 0
  Countries - NaN ID: 0 | Duplicated ID: 0
  Genres   - NaN ID: 0 | Duplicated ID: 0
  Labels   - NaN ID: 0 | Duplicated ID: 0
  Releases - NaN ID: 0 | Duplicated ID: 0

--- Required Literals (names/titles) ---
  Bands without name:     0
  Countries without name: 0
  Genres without name:    0
  Labels without name:    0
  Releases without title: 0

--- Relationship Completeness ---
  Bands with country (hasCountry):   182,277 / 182,277
  Bands with label (producedBy):     31,111 / 182,277
  Bands with releases:               182,277 / 182,277
  Bands with genre (hasGenre):       182,277 / 182,277
  Releases with band (releasedBy):   627,118 / 627,118

--- Foreign Key Validity (Critical) ---
  Bands -> Countries orphan refs:  0
  Bands -> Labels or

| Issue | Count | Action |
|---|---|---|
| Labels -> Bands orphan refs | 276 | Skip triple (label references band removed by dedup) |
| releaseYear "Unknown Year" | 3,769 | Skip `releaseYear` triple for those releases |
| bandStatus values | 6 | Capitalize for individuals: Active, Changed Name, Disputed, On Hold, Split-up, Unknown |
| labelStatus values | 5 | Capitalize for individuals: Active, Changed Name, Closed, On Hold, Unknown |
| releaseType values | 12 | Create `mo:ReleaseType` individuals for each value |